# TopicGPT: Keyword Trend Evaluation (Scenario 4)

1. **Ground Truth**: TF-IDF per year → classify keywords as Emerging / Stable / Decaying
2. **SPAN**: Longest consecutive years each keyword appears in model topics

In [1]:
import pandas as pd
import numpy as np
import ast
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from collections import defaultdict
import warnings
warnings.filterwarnings("ignore")

In [2]:
LIST_SUBJECT = ["cs", "math", "physics"]
DATA_DIR = Path("../../../../data/preprocess")
TEMPORAL_DIR = Path("../../../../results/topicGpt/temporal")
RESULT_DIR = Path("../../../../results/topicGpt/tren")

TOP_K = 20  # top keywords per category
EARLY_YEARS = range(2000, 2011)   # 2000-2010
LATE_YEARS = range(2015, 2026)     # 2015-2025

for subject in LIST_SUBJECT:
    (RESULT_DIR / subject).mkdir(parents=True, exist_ok=True)

print(f"Data: {DATA_DIR}")
print(f"Topics: {TEMPORAL_DIR}")
print(f"Output: {RESULT_DIR}")
print(f"Top-K keywords per category: {TOP_K}")

Data: ../../../../data/preprocess
Topics: ../../../../results/topicGpt/temporal
Output: ../../../../results/topicGpt/tren
Top-K keywords per category: 20


## Paper-Faithful avg-SPAN (Gupta et al., 2018)

Following the SPAN metric from *Deep Temporal-Recurrent-Replicated-Softmax* (arXiv:1711.05626v2):

- **keyword-trend**: binary sequence of keyword appearance in **any** discovered topic per year
- **SPAN (Sₖ)**: length of the longest consecutive 1s in keyword-trend
- **v̂ₖ**: total count of keyword k across **all documents in the corpus** (not just topic words)
- **Sₖ^dict = Sₖ / v̂ₖ**: frequency-normalized SPAN per keyword
- **avg-SPAN = (1/||Q̂||) × Σ Sₖ^dict**: averaged over all unique topic-terms

This computes SPAN over **all** words that appear in discovered topics, normalized by
their corpus frequency to reward models that capture rare-but-trending terms.

In [3]:
def compute_paper_span(keyword, topic_words_by_year, years):
    """
    Compute keyword-trend and SPAN per the paper's definition.
    Returns: span, keyword_trend (list of 0/1)
    """
    trend = []
    for y in years:
        found = any(keyword in words for words in topic_words_by_year.get(y, []))
        trend.append(1 if found else 0)

    # SPAN = longest consecutive 1s
    max_span = 0
    current = 0
    for t in trend:
        if t == 1:
            current += 1
            max_span = max(max_span, current)
        else:
            current = 0

    return max_span, trend



def compute_corpus_word_freq(subject):
    """
    Compute v̂ₖ = total count of each word across ALL documents in the corpus.
    Per paper: v̂ₖ = Σ_{t=1}^{T} Σ_{j=1}^{D_t} v_{j,t}^k
    """
    df = pd.read_csv(DATA_DIR / subject / "bow/v1.csv")
    word_freq = defaultdict(int)
    for text_val in df["text"]:
        try:
            tokens = ast.literal_eval(text_val)
            if isinstance(tokens, list):
                for w in tokens:
                    word_freq[w] += 1
        except (ValueError, SyntaxError):
            for w in str(text_val).split():
                word_freq[w] += 1
    return word_freq

for subject in LIST_SUBJECT:
    print(f"\n{'='*70}")
    print(f"Paper-Faithful avg-SPAN: {subject.upper()} (TopicGPT)")
    print(f"{'='*70}")

    # 1. Load topic-word evolution
    evo_df = pd.read_csv(TEMPORAL_DIR / subject / "topic_word_evolution.csv")
    years = sorted(evo_df['year'].unique())

    topic_words_by_year = defaultdict(list)
    for _, row in evo_df.iterrows():
        words = set(w.strip() for w in str(row['top_words']).split(','))
        topic_words_by_year[int(row['year'])].append(words)

    # 2. Collect ALL unique topic-terms (||Q̂||)
    all_topic_terms = set()
    for year_words_list in topic_words_by_year.values():
        for word_set in year_words_list:
            all_topic_terms.update(word_set)

    # 3. Compute corpus word frequency (v̂ₖ from documents)
    print(f"  Computing corpus word frequencies...")
    corpus_freq = compute_corpus_word_freq(subject)
    print(f"  Corpus vocabulary: {len(corpus_freq)} unique words")
    print(f"  Topic-terms (||Q̂||): {len(all_topic_terms)}")

    # 4. Compute SPAN and Sₖ^dict for every topic-term
    paper_rows = []
    for word in sorted(all_topic_terms):
        span, trend = compute_paper_span(word, topic_words_by_year, years)
        v_hat = corpus_freq.get(word, 0)  # corpus frequency
        s_dict = span / v_hat if v_hat > 0 else 0.0

        paper_rows.append({
            'word': word,
            'span': span,
            'v_hat': v_hat,
            's_dict': round(s_dict, 6),
            'keyword_trend': str(trend),
            'total_years': len(years),
            'years_present': sum(trend),
            'coverage_pct': round(sum(trend) / len(years) * 100, 1),
        })

    paper_df = pd.DataFrame(paper_rows)
    paper_df.to_csv(RESULT_DIR / subject / 'keyword_span_paper.csv', index=False)

    # 5. avg-SPAN (paper formula): (1/||Q̂||) × Σ Sₖ^dict
    q_hat = len(all_topic_terms)
    sum_s_dict = paper_df['s_dict'].sum()
    avg_span_paper = sum_s_dict / q_hat if q_hat > 0 else 0.0
    avg_span_simple = paper_df['span'].mean()

    summary = {
        'subject': subject,
        'model': 'TopicGPT',
        'total_unique_terms': q_hat,
        'avg_span_paper': round(avg_span_paper, 6),
        'avg_span_simple': round(avg_span_simple, 4),
        'sum_s_dict': round(sum_s_dict, 6),
        'total_corpus_freq': int(paper_df['v_hat'].sum()),
        'terms_not_in_corpus': int((paper_df['v_hat'] == 0).sum()),
    }
    pd.DataFrame([summary]).to_csv(RESULT_DIR / subject / 'keyword_span_paper_summary.csv', index=False)

    print(f"  avg-SPAN (paper, freq-normalized):  {avg_span_paper:.6f}")
    print(f"  avg-SPAN (simple mean):             {avg_span_simple:.4f}")
    print(f"  Total corpus freq (Σv̂ₖ):            {int(paper_df['v_hat'].sum())}")
    print(f"  Topic-terms not in corpus:          {(paper_df['v_hat'] == 0).sum()}")

    # Top-10 by SPAN
    print(f"\n  Top 10 by SPAN:")
    print(f"  {'Word':<25s} {'SPAN':>5s} {'v̂ₖ':>8s} {'Sₖ^dict':>10s}")
    for _, r in paper_df.nlargest(10, 'span').iterrows():
        print(f"  {r['word']:<25s} {r['span']:>5d} {r['v_hat']:>8d} {r['s_dict']:>10.6f}")

    # Top-10 by Sₖ^dict (highest frequency-normalized SPAN — rare but persistent)
    print(f"\n  Top 10 by Sₖ^dict (rare but persistent):")
    for _, r in paper_df[paper_df['v_hat'] > 0].nlargest(10, 's_dict').iterrows():
        print(f"  {r['word']:<25s} {r['span']:>5d} {r['v_hat']:>8d} {r['s_dict']:>10.6f}")

    print(f"\n  Saved: {RESULT_DIR / subject}")



Paper-Faithful avg-SPAN: CS (TopicGPT)
  Computing corpus word frequencies...
  Corpus vocabulary: 146603 unique words
  Topic-terms (||Q̂||): 4071
  avg-SPAN (paper, freq-normalized):  0.090211
  avg-SPAN (simple mean):             2.7219
  Total corpus freq (Σv̂ₖ):            7188469
  Topic-terms not in corpus:          0

  Top 10 by SPAN:
  Word                       SPAN      v̂ₖ    Sₖ^dict
  algorithm                    26    81182   0.000320
  document                     26     7884   0.003298
  graph                        26    51348   0.000506
  language                     26    46147   0.000563
  logic                        26     8543   0.003043
  program                      26    10099   0.002575
  agent                        25    27807   0.000899
  code                         25    29942   0.000835
  datum                        25   116032   0.000215
  dominating                   25      457   0.054705

  Top 10 by Sₖ^dict (rare but persistent):
  attempting   